# C10-competition-craft — Practice p13 — Solution

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

SEED = 20260804
ks = np.array([5, 7, 9, 11])

df = pd.read_csv("../data/train.csv")
FEATURES = [c for c in df.columns if c != "outcome"]
X = df[FEATURES]
y = df["outcome"].to_numpy()
class_counts = {label: int(count) for label, count in df["outcome"].value_counts().items()}

X_tr, X_val, y_tr, y_val = train_test_split(
    X, y, test_size=150, random_state=SEED, stratify=y
)
base = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier(n_neighbors=5)),
]).fit(X_tr, y_tr)
base_preds = base.predict(X_val)
val_f1_base = float(f1_score(y_val, base_preds, average="macro"))

C_val = confusion_matrix(y_val, base_preds)
diag = np.diag(C_val).astype(float)
prec = diag / C_val.sum(axis=0)
rec = diag / C_val.sum(axis=1)
per_class_f1 = 2 * prec * rec / (prec + rec)
labels = np.unique(y_val)
weaker_class = str(labels[np.argmin(per_class_f1)])

scores = []
for k in ks:
    candidate = Pipeline([
        ("scaler", StandardScaler()),
        ("knn", KNeighborsClassifier(n_neighbors=int(k))),
    ]).fit(X_tr, y_tr)
    scores.append(f1_score(y_val, candidate.predict(X_val), average="macro"))
val_f1s = np.array(scores, dtype=float)
best_k = int(ks[np.argmax(val_f1s)])

final_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier(n_neighbors=best_k)),
]).fit(X, y)


def predict_labels(X_test):
    return pd.Series(final_pipe.predict(X_test), index=X_test.index)


probe = X.iloc[100:140]
probe_out = predict_labels(probe)
contract_ok = bool(
    isinstance(probe_out, pd.Series)
    and len(probe_out) == len(probe)
    and probe_out.index.equals(probe.index)
    and set(probe_out.unique()) <= set(np.unique(y))
)
(class_counts, val_f1_base, C_val, weaker_class, val_f1s, best_k, contract_ok)

The confusion-piece calculation identifies `struggles` as the weaker class. The one-change sweep keeps the same carve, features, scaler, and macro-F1 judge, isolating the effect of (k).

### Answer check

In [ ]:
assert X.shape == (600, 12) and y.shape == (600,)
assert class_counts == {"thrives": 379, "struggles": 221}
assert np.isclose(val_f1_base, 0.7665823769694612, atol=1e-12, rtol=0)
assert np.array_equal(C_val, np.array([[37, 18], [14, 81]]))
assert weaker_class == "struggles"
expected = np.array([0.7665823769694612, 0.7939560439560439,
                     0.7893564476296547, 0.8103481812876873])
assert val_f1s.shape == (4,)
assert np.allclose(val_f1s, expected, atol=1e-12, rtol=0)
assert best_k == 11
assert contract_ok is True